# GA + Breakout animado  
*   Elemento de lista
*   Elemento de lista



Esta celda implementa un **mini-Breakout** y un **Algoritmo Genético (GA)** que aprende a jugarlo. Al ejecutarla verás:

- **Animación** de episodios completos para **Gen 0, 1, 2** y la **mejor global** (según validación).
- **Curvas** de rendimiento:
  - **TRAIN fijo**: mejor puntuación por generación con semillas fijas (comparables).
  - **Acumulado TRAIN**: máximo histórico de TRAIN (no decrece).
  - **VALIDACIÓN fija**: récord con semillas fijas (no decrece, referencia de progreso real).

In [ ]:
# Breakout + Algoritmo Genético (GA) + Animación "episodio completo por generación"
# ------------------------------------------------------------------------------
# FITNESS = SOLO BLOQUES ROTOS
# - Preserva intactos: campeón histórico (validación) + campeón gen previo (train fijo)
# - Log compacto por eventos; en el log "best=" MUESTRA # DE BLOQUES ROTOS (entero)
# - Animación: Gen 0, 1, 2 y la MEJOR GLOBAL; la mejor corre con más pasos.
# - Overlay: Bloques actuales y Mejor (máximo entre episodios mostrados).
# - COLAB: se guarda como MP4 externo y se muestra sin incrustarlo en el notebook

import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle, Circle
from IPython.display import Video, display

matplotlib.rcParams['animation.html'] = 'html5'

# =========================
# ENTORNO: Breakout mínimo
# =========================
class BreakoutEnv:
    def __init__(
        self, W=420, H=300, rows=4, cols=6,
        paddle_w=64, paddle_h=8, paddle_speed=6.0,
        ball_r=4.0, ball_speed=4.5, brick_h=16,
        left_margin=20, top_margin=40, seed=None
    ):
        self.W, self.H = W, H
        self.rows, self.cols = rows, cols
        self.paddle_w, self.paddle_h = paddle_w, paddle_h
        self.paddle_speed = paddle_speed
        self.ball_r = ball_r
        self.ball_speed = ball_speed
        self.brick_h = brick_h
        self.left_margin, self.top_margin = left_margin, top_margin
        self.rng = np.random.default_rng(seed)
        self.brick_w = (self.W - 2*self.left_margin)/self.cols
        self.paddle_y = self.H - 18
        self.reset(seed=seed)

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.bricks = np.ones((self.rows, self.cols), dtype=bool)
        self.total_bricks = self.rows * self.cols
        self.bricks_broken = 0
        self.paddle_x = self.W/2.0
        self.ball_x = self.W/2.0
        self.ball_y = self.paddle_y - 14.0
        vx = self.rng.uniform(-0.7, 0.7) * self.ball_speed
        vy = -np.sqrt(max(self.ball_speed**2 - vx**2, 1e-6))
        self.vx, self.vy = vx, vy
        self.t = 0
        return self._obs()

    def _obs(self):
        x = self.ball_x / self.W
        y = self.ball_y / self.H
        vx = self.vx / self.ball_speed
        vy = self.vy / self.ball_speed
        px = self.paddle_x / self.W
        dx = (self.ball_x - self.paddle_x) / self.W
        return np.array([x, y, vx, vy, px, dx], dtype=np.float32)

    def _bounce_walls(self):
        r = self.ball_r
        if self.ball_x < r:
            self.ball_x = r
            self.vx = abs(self.vx)
        elif self.ball_x > self.W - r:
            self.ball_x = self.W - r
            self.vx = -abs(self.vx)
        if self.ball_y < r:
            self.ball_y = r
            self.vy = abs(self.vy)

    def _paddle_collision(self):
        half = self.paddle_w/2.0
        if (self.ball_y + self.ball_r >= self.paddle_y - self.paddle_h/2.0 and
            self.ball_y - self.ball_r <= self.paddle_y + self.paddle_h/2.0 and
            self.ball_x >= self.paddle_x - half - self.ball_r and
            self.ball_x <= self.paddle_x + half + self.ball_r and
            self.vy > 0):
            offset = (self.ball_x - self.paddle_x) / half
            self.vy = -abs(self.vy)
            self.vx += 1.5 * offset
            v = np.hypot(self.vx, self.vy)
            if v > self.ball_speed * 1.3:
                s = (self.ball_speed * 1.3) / v
                self.vx *= s
                self.vy *= s

    def _brick_collision(self):
        bx = int((self.ball_x - self.left_margin) // self.brick_w)
        by = int((self.ball_y - self.top_margin) // self.brick_h)
        if 0 <= bx < self.cols and 0 <= by < self.rows:
            if self.bricks[by, bx]:
                self.bricks[by, bx] = False
                self.bricks_broken += 1
                self.vy *= -1.0

    def step(self, action):
        # action ∈ {-1, 0, 1}
        self.paddle_x += action * self.paddle_speed
        self.paddle_x = np.clip(self.paddle_x, self.paddle_w/2.0, self.W - self.paddle_w/2.0)

        self.ball_x += self.vx
        self.ball_y += self.vy

        self._bounce_walls()
        self._brick_collision()
        self._paddle_collision()

        self.t += 1

        done = self.ball_y > self.H + self.ball_r
        if not self.bricks.any():  # todos rotos
            done = True
        return self._obs(), done


# =========================
# POLÍTICA LINEAL
# =========================
F = 6
A = 3
PARAM_DIM = A*F + A

def split_params(theta):
    W = theta[:A*F].reshape(A, F)
    b = theta[A*F:].reshape(A,)
    return W, b

def act(theta, feat):
    W, b = split_params(theta)
    z = W @ feat + b
    a = int(np.argmax(z))
    return (-1, 0, 1)[a]


# =========================
# EVALUACIÓN (fitness = bloques rotos)
# =========================
def rollout_score(theta, max_steps=2000, seed=0):
    env = BreakoutEnv(seed=seed)
    env.reset(seed=seed)
    for _ in range(max_steps):
        feat = env._obs()
        a = act(theta, feat)
        _, done = env.step(a)
        if done:
            break
    return float(env.bricks_broken)

def eval_individual(theta, episodes=2, base_seed=0):
    scores = [rollout_score(theta, seed=base_seed + k) for k in range(episodes)]
    return float(np.mean(scores))


# =========================
# GA helpers
# =========================
def init_pop(n=120, scale=0.5, seed=0):
    rng = np.random.default_rng(seed)
    return rng.normal(0.0, scale, size=(n, PARAM_DIM))

def selection(pop, fitness, elite=20, k_torneo=3, extra_random=8, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    idx = np.argsort(-fitness)
    elites = pop[idx[:elite]].copy()
    padres = [p for p in elites]
    N = len(pop)
    for _ in range(N - len(padres) - extra_random):
        cand = rng.integers(0, N, size=k_torneo)
        ganador = cand[np.argmax(fitness[cand])]
        padres.append(pop[ganador].copy())
    for _ in range(extra_random):
        padres.append(pop[rng.integers(0, N)].copy())
    rng.shuffle(padres)
    return np.array(padres), elites

def crossover_blend(p1, p2, rng=None, alpha=0.5):
    rng = np.random.default_rng() if rng is None else rng
    u = rng.random(size=p1.shape)
    return alpha*u*p1 + (1-alpha*u)*p2

def mutate(theta, sigma=0.15, p_mut=0.4, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    m = theta.copy()
    mask = rng.random(size=m.shape) < p_mut
    m[mask] += rng.normal(0.0, sigma, size=mask.sum())
    return m

def breed_keep_elites(champion, elite2, pool, total, sigma=0.15, p_mut=0.4, rng=None, self_mate_prob=0.2):
    """Nueva población con élites intactos: pop[0]=campeón histórico; pop[1]=mejor gen pasada (si distinto)."""
    rng = np.random.default_rng() if rng is None else rng
    elites = [champion]
    if elite2 is not None and not np.array_equal(elite2, champion):
        elites.append(elite2)
    children_needed = total - len(elites)
    children = []
    pool = list(pool)
    if len(pool) == 0:
        pool = [mutate(champion, sigma=sigma, p_mut=1.0, rng=rng)]
    i = j = 0
    while len(children) < children_needed:
        parent_main = elites[i % len(elites)]
        i += 1
        mate = parent_main if rng.random() < self_mate_prob else pool[j % len(pool)]
        if mate is not parent_main:
            j += 1
        child = crossover_blend(parent_main, mate, rng=rng)
        child = mutate(child, sigma=sigma, p_mut=p_mut, rng=rng)
        children.append(child)
    return np.vstack([np.vstack(elites), np.array(children)])


# =========================
# ENTRENAMIENTO (anti-estancamiento + TRAIN fijo + VALIDACIÓN fija)
# =========================
def train_GA_antistagnation(
    gens=50, pop_size=50, elite=10,
    sigma=0.22, p_mut=0.2,
    seed=2025,
    patience=6, min_delta=1e-3,
    burst_len=5, sigma_mult=3.0, p_mut_bump=0.30,
    immigrants_frac=0.35, elite_shrink=0.5, k_torneo_burst=2,
    # Validación fija (misma métrica: bloques rotos)
    val_base_seed=12345, val_episodes=8,
    # TRAIN fijo (comparabilidad total)
    train_fixed_seed=777, episodes_eval_train=5,
    # Log
    log_mode="events", log_every=5, verbose=True
):
    rng = np.random.default_rng(seed)
    pop = init_pop(n=pop_size, scale=0.6, seed=seed)

    best_per_gen = []
    best_train_curve = []
    best_train_curve_running_max = []
    best_val_curve = []

    champion_hist = None
    best_score_global_val = -1e9
    burst_timer = 0
    no_improve = 0
    running_max = -1e-9
    best_prev_gen = None

    def eval_val(theta):
        return eval_individual(theta, episodes=val_episodes, base_seed=val_base_seed)

    for g in range(gens):
        base_seed_train = train_fixed_seed
        fitness = np.array([
            eval_individual(ind, episodes=episodes_eval_train, base_seed=base_seed_train)
            for ind in pop
        ])
        idx_best = int(np.argmax(fitness))
        best_curr = pop[idx_best].copy()
        best_train = float(fitness[idx_best])

        # bloques rotos (entero) del mejor con la semilla fija de TRAIN
        best_blocks_train = int(rollout_score(best_curr, seed=base_seed_train))

        cand_val = eval_val(best_curr)
        new_val_record = (cand_val > best_score_global_val + min_delta)
        new_train_peak = (best_train > running_max + 1e-12)

        # LOG compacto por eventos (best en BLOQUES)
        if verbose:
            if log_mode == "events":
                if g == 0 or g == gens-1 or new_train_peak or new_val_record:
                    shown_val = cand_val if new_val_record else best_score_global_val
                    tags = []
                    if new_train_peak:
                        tags.append("TRAIN▲")
                    if new_val_record:
                        tags.append("VAL▲")
                    if burst_timer > 0:
                        tags.append("BURST")
                    tag_str = (" [" + " ".join(tags) + "]") if tags else ""
                    print(f"Gen {g:02d} | best={best_blocks_train} | val={shown_val:.3f}{tag_str}")
            else:
                if (g % log_every == 0) or (g == gens-1):
                    shown_val = cand_val if new_val_record else best_score_global_val
                    print(f"Gen {g:02d} | best={best_blocks_train} | val={shown_val:.3f}")

        # Récord VALIDACIÓN (campeón histórico)
        if new_val_record:
            best_score_global_val = cand_val
            champion_hist = best_curr.copy()
            no_improve = 0
        else:
            no_improve += 1
            if champion_hist is None:
                champion_hist = best_curr.copy()

        # Curvas
        best_per_gen.append(best_curr.copy())
        best_train_curve.append(best_train)
        running_max = max(running_max, best_train)
        best_train_curve_running_max.append(running_max)
        best_val_curve.append(best_score_global_val)

        # Anti-estancamiento
        if no_improve >= patience and burst_timer == 0:
            burst_timer = burst_len
        if burst_timer > 0:
            sigma_eff = sigma * sigma_mult
            p_mut_eff = min(0.95, p_mut + p_mut_bump)
            elite_eff = max(1, int(elite * elite_shrink))
            k_torneo_eff = k_torneo_burst
        else:
            sigma_eff = sigma * (0.98 ** g)
            p_mut_eff = p_mut
            elite_eff = elite
            k_torneo_eff = 3

        # Selección y reproducción con élites intactos
        padres, _ = selection(pop, fitness, elite=elite_eff,
                              k_torneo=k_torneo_eff, extra_random=8, rng=rng)
        elite2 = None
        if best_prev_gen is not None and not np.array_equal(best_prev_gen, champion_hist):
            elite2 = best_prev_gen
        pool = [ind for ind in padres
                if (not np.array_equal(ind, champion_hist) and (elite2 is None or not np.array_equal(ind, elite2)))]
        pop = breed_keep_elites(champion=champion_hist, elite2=elite2, pool=pool, total=pop_size,
                                sigma=sigma_eff, p_mut=p_mut_eff, rng=rng, self_mate_prob=0.2)

        assert np.array_equal(pop[0], champion_hist), "El campeón histórico NO quedó intacto en pop[0]."
        if elite2 is not None:
            assert any(np.array_equal(ind, elite2) for ind in pop), "El best de la gen pasada no fue preservado."

        best_prev_gen = best_curr.copy()

        # Inmigración (no toca élites)
        if burst_timer > 0 and immigrants_frac > 0:
            num_elites_fijos = 1 + (0 if elite2 is None else 1)
            num_imm = int(max(0, (pop_size - num_elites_fijos) * immigrants_frac))
            if num_imm > 0:
                immigrants = init_pop(n=num_imm, scale=0.8, seed=seed + 999 + g)
                pop[-num_imm:] = immigrants

        if burst_timer > 0:
            burst_timer -= 1
            if new_val_record:
                burst_timer = 0

    return (np.array(best_per_gen, dtype=object),
            np.array(best_train_curve),
            np.array(best_train_curve_running_max),
            np.array(best_val_curve))


# =========================
# FRAMES y ANIMACIÓN
# =========================
def select_first_n_plus_best_global(best_per_gen, fitness_curve_val, first_n=3):
    total = len(best_per_gen)
    primeros = list(range(min(first_n, total)))
    idx_best_global = int(np.argmax(fitness_curve_val)) if len(fitness_curve_val) else 0
    indices = primeros[:]
    if idx_best_global not in indices:
        indices.append(idx_best_global)
    return indices, idx_best_global

def full_episode_frames_selected(best_per_gen, gen_indices, max_steps=800,
                                 best_global_idx=None, max_steps_best=3000,
                                 frame_stride=2):
    """
    Episodios completos; si es la mejor global, usa más pasos para no truncar.
    Añade 'best_blocks' = máximo de bloques entre los episodios seleccionados.
    frame_stride=2 guarda 1 frame cada 2 pasos para bajar el peso del video
    sin alterar demasiado el estilo visual original.
    """
    env0 = BreakoutEnv()
    rows, cols = env0.rows, env0.cols

    episodes = []
    for g in gen_indices:
        theta = best_per_gen[g]
        env = BreakoutEnv(seed=100 + g)
        env.reset(seed=100 + g)
        tag = "MEJOR GLOBAL" if (best_global_idx is not None and g == best_global_idx) else ""
        limit = max_steps_best if (best_global_idx is not None and g == best_global_idx) else max_steps

        epi_frames = []
        for step in range(limit):
            feat = env._obs()
            a = act(theta, feat)
            _, done = env.step(a)

            if (step % frame_stride == 0) or done:
                epi_frames.append({
                    "gen": g,
                    "step": step,
                    "ball": (env.ball_x, env.ball_y),
                    "paddle": env.paddle_x,
                    "bricks": env.bricks.flatten().copy(),
                    "blocks": env.bricks_broken,
                    "tag": tag
                })

            if done:
                break

        episodes.append({"frames": epi_frames, "final_blocks": env.bricks_broken})

    best_blocks_global = max(ep["final_blocks"] for ep in episodes) if episodes else 0
    frames = []
    for ep in episodes:
        for fr in ep["frames"]:
            fr["best_blocks"] = int(best_blocks_global)
            frames.append(fr)

    return frames, rows, cols

def animate_breakout(frames, rows, cols,
                     W=420, H=300, left_margin=20, top_margin=40, brick_h=16,
                     interval_ms=60, figsize=(7.0, 5.0)):
    brick_w = (W - 2*left_margin)/cols
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, W)
    ax.set_ylim(0, H)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title("GA aprendiendo Breakout — Episodio completo por generación")

    bricks_patches = []
    for r in range(rows):
        for c in range(cols):
            x0 = left_margin + c*brick_w
            y0 = top_margin + r*brick_h
            rect = Rectangle((x0, y0), brick_w-1.0, brick_h-1.0, fc="#f39c12", ec="#d35400")
            ax.add_patch(rect)
            bricks_patches.append(rect)

    paddle = Rectangle((W/2 - 32, H-18 - 4), 64, 8, fc="#2980b9", ec="#1f5f85")
    ax.add_patch(paddle)
    ball = Circle((W/2, H/2), radius=4.0, fc="#2ecc71", ec="#1e8449")
    ax.add_patch(ball)
    txt = ax.text(0.02, 0.98, "", transform=ax.transAxes, va='top', ha='left',
                  bbox=dict(boxstyle='round', fc='white', ec='0.8', alpha=0.9), fontsize=10)

    def init():
        return bricks_patches + [paddle, ball, txt]

    def update(i):
        fr = frames[i]
        alive = fr["bricks"]
        for k, rect in enumerate(bricks_patches):
            rect.set_visible(bool(alive[k]))
        paddle.set_x(fr["paddle"] - paddle.get_width()/2.0)
        ball.center = fr["ball"]
        tag = f"   [{fr['tag']}]" if fr.get('tag') else ""
        txt.set_text(f"Gen: {fr['gen']:02d}   Step: {fr['step']:04d}   "
                     f"Bloques: {int(fr['blocks'])}   Mejor: {int(fr['best_blocks'])}{tag}")
        return bricks_patches + [paddle, ball, txt]

    anim = animation.FuncAnimation(fig, update, init_func=init,
                                   frames=len(frames), interval=interval_ms, blit=True)
    plt.close(fig)
    return anim


# =========================
# COLAB: GUARDAR Y MOSTRAR VIDEO SIN INCRUSTARLO EN EL NOTEBOOK
# =========================
def guardar_animacion_colab(anim, filename="breakout_ga.mp4", fps=20, dpi=90):
    """
    Guarda la animación como MP4 externo.
    Esto mantiene el estilo original, pero evita inflar el notebook
    como ocurría con to_jshtml().
    """
    try:
        writer = animation.FFMpegWriter(fps=fps)
        anim.save(filename, writer=writer, dpi=dpi)
        print(f"Animación guardada como: {filename}")
        return filename
    except Exception as e:
        alt = filename.replace(".mp4", ".gif")
        print("No se pudo guardar en MP4 con ffmpeg.")
        print("Motivo:", e)
        print("Guardando como GIF...")
        writer = animation.PillowWriter(fps=min(fps, 15))
        anim.save(alt, writer=writer, dpi=dpi)
        print(f"Animación guardada como: {alt}")
        return alt

def mostrar_animacion_colab(path, width=720):
    """
    Muestra el archivo generado sin incrustarlo en el notebook.
    En Colab esto ayuda a que el .ipynb no se dispare de tamaño.
    """
    if path.lower().endswith(".mp4"):
        display(Video(path, embed=False, width=width))
    else:
        from IPython.display import Image
        display(Image(filename=path, width=width))


# =========================
# ENTRENAR, SELECCIONAR Y MOSTRAR
# =========================
best_per_gen, fitness_train_curve, fitness_train_running_max, fitness_val_curve = train_GA_antistagnation(
    gens=50, pop_size=50, elite=10,
    sigma=0.22, p_mut=0.2,
    seed=2025,
    patience=6, min_delta=1e-3, burst_len=5,
    sigma_mult=3.0, p_mut_bump=0.30, immigrants_frac=0.35,
    elite_shrink=0.5, k_torneo_burst=2,
    train_fixed_seed=777, episodes_eval_train=5,
    val_base_seed=12345, val_episodes=8,
    log_mode="events", log_every=5, verbose=True
)

# Mantengo tu estilo original: primeras generaciones + mejor global
sel_idx, idx_best_global = select_first_n_plus_best_global(best_per_gen, fitness_val_curve, first_n=2)
print("Generaciones mostradas:", sel_idx, "| Mejor global (validación):", idx_best_global)

frames, rows, cols = full_episode_frames_selected(
    best_per_gen,
    sel_idx,
    max_steps=400,
    best_global_idx=idx_best_global,
    max_steps_best=3000,
    frame_stride=2
)

anim = animate_breakout(
    frames, rows, cols,
    interval_ms=10,
    figsize=(7.0, 5.0)
)

# En Colab: guardar fuera del notebook y luego mostrar
video_path = guardar_animacion_colab(anim, filename="breakout_ga.mp4", fps=20, dpi=90)
mostrar_animacion_colab(video_path, width=720)

plt.figure(figsize=(8.6, 3.2))
plt.plot(fitness_train_curve,       label="Mejor por gen (TRAIN fijo) — bloques", lw=2)
plt.plot(fitness_train_running_max, label="Acumulado monótono (TRAIN) — bloques", lw=2)
plt.plot(fitness_val_curve,         label="Récord acumulado (VALIDACIÓN) — bloques", lw=2)
plt.xlabel("Generación")
plt.ylabel("Bloques rotos")
plt.title("Evolución del fitness (solo bloques rotos)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

Gen 00 | best=1 | val=3.625 [TRAIN▲ VAL▲]
Gen 01 | best=1 | val=3.625 [TRAIN▲]
Gen 03 | best=2 | val=3.625 [TRAIN▲]
Gen 20 | best=5 | val=3.625 [TRAIN▲ BURST]
Gen 24 | best=7 | val=3.625 [TRAIN▲ BURST]
Gen 29 | best=3 | val=9.375 [VAL▲ BURST]
Gen 31 | best=21 | val=9.375 [TRAIN▲]
Gen 32 | best=19 | val=9.375 [TRAIN▲]
Gen 34 | best=24 | val=9.375 [TRAIN▲]
Gen 35 | best=16 | val=9.375 [TRAIN▲]
Gen 39 | best=20 | val=19.875 [TRAIN▲ VAL▲ BURST]
Gen 41 | best=20 | val=19.875 [TRAIN▲]
Gen 42 | best=23 | val=21.375 [TRAIN▲ VAL▲]
Gen 43 | best=24 | val=21.375 [TRAIN▲]
Gen 44 | best=24 | val=21.375 [TRAIN▲]
Gen 45 | best=23 | val=21.375 [TRAIN▲]
Gen 46 | best=24 | val=21.375 [TRAIN▲]
